# Camera Analysis — Ground Truth Config

Fetches one image per LTA camera, extracts text overlays using EasyOCR,
and produces `camera_config.json` — the ground truth for the road network.

Run once. Output pushed to HF Hub as `camera_config.json`.

**What we extract per camera:**
- Road name (from text label)
- All visible directions (e.g. 'towards Changi', 'towards Tuas')
- Whether it's a junction/interchange camera
- Number of visible lanes (from text if available, else from image)
- Raw OCR text for manual verification

In [ ]:
!pip install -q easyocr huggingface_hub requests Pillow

In [ ]:
import requests, json, urllib.request, re, time
from io import BytesIO
from PIL import Image
import numpy as np
import easyocr

LTA_API = 'https://api.data.gov.sg/v1/transport/traffic-images'

# Fetch all cameras
r = requests.get(LTA_API, timeout=15)
cameras = r.json()['items'][0]['cameras']
print(f'Found {len(cameras)} cameras')

# Init OCR (English only, CPU)
reader = easyocr.Reader(['en'], gpu=False, verbose=False)
print('OCR ready')

In [ ]:
def fetch_image(url):
    try:
        req = urllib.request.Request(url, headers={
            'User-Agent': 'Mozilla/5.0',
            'Referer': 'https://data.gov.sg/'
        })
        data = urllib.request.urlopen(req, timeout=8).read()
        return Image.open(BytesIO(data)).convert('RGB')
    except Exception as e:
        return None


def extract_text(image):
    """Extract all text from top 15% and bottom 10% of image."""
    w, h = image.size
    texts = []
    for crop in [
        image.crop((0, 0, w, int(h * 0.15))),       # top strip
        image.crop((0, int(h * 0.90), w, h)),         # bottom strip
        image.crop((0, int(h * 0.15), int(w*0.3), int(h * 0.40))),  # top-left (road name)
    ]:
        results = reader.readtext(np.array(crop), detail=0)
        texts.extend([t.strip() for t in results if t.strip()])
    return texts


def parse_directions(texts):
    """Parse direction labels from OCR text."""
    directions = []
    patterns = [
        r'towards?\s+([A-Za-z][A-Za-z\s]{2,25})(?:\s*$|\s+\d|\s+\()',
        r'to\s+([A-Za-z][A-Za-z\s]{2,25})(?:\s*$|\s+\d)',
        r'([A-Za-z][A-Za-z\s]{2,20})\s*(?:bound|BND|BOUND)',
    ]
    for text in texts:
        for p in patterns:
            m = re.search(p, text, re.IGNORECASE)
            if m:
                label = m.group(1).strip().title()
                full = f'towards {label}'
                if full not in directions and len(label) > 2:
                    directions.append(full)
    return directions


def parse_road(texts, cam_id):
    """Extract road name from OCR text."""
    road_codes = ['PIE', 'CTE', 'AYE', 'ECP', 'MCE', 'TPE', 'BKE', 'KJE', 'SLE']
    for text in texts:
        for code in road_codes:
            if re.search(r'\b' + code + r'\b', text, re.IGNORECASE):
                return code
    # Fallback from camera ID prefix
    mce_ids = {'6702','6703','6704','6705'}
    prefix_map = {'1':'CTE','2':'CTE','3':'ECP','4':'PIE','5':'AYE',
                  '6':'ECP','7':'TPE','8':'KJE','9':'BKE'}
    if cam_id in mce_ids:
        return 'MCE'
    return prefix_map.get(cam_id[0], '—')


def parse_lane_count(texts):
    """Try to extract explicit lane count from text."""
    for text in texts:
        m = re.search(r'(\d)\s*lane', text, re.IGNORECASE)
        if m:
            return int(m.group(1))
    return None  # unknown — will use Hough detection


print('Functions ready')

In [ ]:
# ── Main analysis loop ────────────────────────────────────────────────────────
camera_config = {}
errors = []

for i, cam in enumerate(cameras):
    cam_id = str(cam['camera_id'])
    loc = cam.get('location', {})
    lat = float(loc.get('latitude', 0))
    lon = float(loc.get('longitude', 0))
    img_url = cam.get('image', '')

    img = fetch_image(img_url)
    if img is None:
        errors.append(cam_id)
        print(f'  [{i+1:02d}/{len(cameras)}] {cam_id} — image failed')
        continue

    texts = extract_text(img)
    road = parse_road(texts, cam_id)
    directions = parse_directions(texts)
    lane_count = parse_lane_count(texts)
    is_junction = len(directions) > 2

    camera_config[cam_id] = {
        'road': road,
        'lat': lat,
        'lon': lon,
        'ocr_text': texts,
        'directions': directions,
        'lane_count_from_text': lane_count,
        'is_junction': is_junction,
        'img_url': img_url,
    }

    print(f'[{i+1:02d}/{len(cameras)}] {cam_id} {road:4s} | dirs: {directions} | lanes: {lane_count} | junction: {is_junction}')

print(f'\nDone. {len(camera_config)} cameras analysed, {len(errors)} errors.')
if errors:
    print(f'Failed: {errors}')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
import pandas as pd

rows = []
for cam_id, cfg in camera_config.items():
    rows.append({
        'camera_id': cam_id,
        'road': cfg['road'],
        'n_directions': len(cfg['directions']),
        'is_junction': cfg['is_junction'],
        'lane_count': cfg['lane_count_from_text'],
        'directions': ' | '.join(cfg['directions']),
        'ocr_text': ' | '.join(cfg['ocr_text']),
    })

df = pd.DataFrame(rows)
print('=== Cameras by road ===')
print(df.groupby('road').size())
print('\n=== Junction cameras ===')
print(df[df['is_junction']].to_string(index=False))
print('\n=== Cameras with lane count in text ===')
print(df[df['lane_count'].notna()].to_string(index=False))
print('\n=== All cameras ===')
print(df.to_string(index=False))

In [ ]:
# ── Save + push to HF Hub ─────────────────────────────────────────────────────
from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID  = 'SuhxsReddy/cati-singapore-dataset'

# Save locally
with open('/content/camera_config.json', 'w') as f:
    json.dump(camera_config, f, indent=2)
print('Saved camera_config.json')

# Push to HF Hub
api = HfApi()
api.upload_file(
    path_or_fileobj='/content/camera_config.json',
    path_in_repo='camera_config.json',
    repo_id=REPO_ID,
    repo_type='dataset',
    token=HF_TOKEN,
)
print(f'Pushed to https://huggingface.co/datasets/{REPO_ID}')